<a href="https://colab.research.google.com/github/luisrd07/FDG2026II-EVIDENCIAS/blob/main/LAB_D2_FGD_2026II_ESTUDIANTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color:#1E3A5F">LABORATORIO DIRIGIDO N.° 02</h1>

**Semana 2 · Fundamentos de Gestión de Datos · 2026-II** · Docente: Pilar Rocío Sayán Mejía

**Tema:** Consultas SQLite para responder preguntas de negocio

**Laboratorio:** Primer reporte comercial de TecnoAndina Retail con SQLite

**Cómo trabajar en este cuaderno.** Ejecute las celdas de código en orden con el botón ▶. Para escribir una respuesta, haga **doble clic** en la celda de texto, escriba y presione **Ctrl + Enter**. Al terminar, el cuaderno ejecutado se deposita en el repositorio del equipo en GitHub, dentro de la carpeta `semana02/`, y el enlace se registra en la actividad de la Semana 2 del aula virtual.

<h2 style="color:#1E3A5F">Actividad 1 — Revisión de conceptos</h2>

**Instrucciones:** complete la tabla con definiciones elaboradas con sus propias palabras. No se admite la reproducción literal de fuentes externas ni de sistemas generativos. La última columna contiene una pregunta de apoyo: si su definición permite responderla, la definición es suficiente; si no lo permite, corríjala antes de continuar.

| Concepto | Definición con sus propias palabras | Pregunta de apoyo |
|---|---|---|
| **Base de datos relacional, tabla, registro y campo** | &nbsp; | <span style="color:#1F4E9C">En la tabla ventas, ¿qué representa una fila y qué representa una columna?</span> |
| **Consulta SQL y SELECT** | &nbsp; | <span style="color:#1F4E9C">¿Cuándo conviene SELECT * y cuándo conviene nombrar las columnas?</span> |
| **WHERE** | &nbsp; | <span style="color:#1F4E9C">Una venta que no cumple la condición del WHERE, ¿se elimina de la tabla? Justifique.</span> |
| **AND, OR y NOT** | &nbsp; | <span style="color:#1F4E9C">Si la gerencia pide ventas de Lima o de Piura, ¿qué operador corresponde y por qué AND no devolvería filas?</span> |
| **ORDER BY y LIMIT** | &nbsp; | <span style="color:#1F4E9C">Para obtener «las cinco ventas más altas», ¿por qué el orden debe definirse antes del límite?</span> |
| **DISTINCT** | &nbsp; | <span style="color:#1F4E9C">Una consulta devuelve la sede repetida muchas veces. ¿Qué cláusula usaría y qué obtendría?</span> |
| **COUNT y SUM** | &nbsp; | <span style="color:#1F4E9C">¿En qué se diferencia contar las ventas de una sede de sumar su monto? Dé un ejemplo del caso.</span> |
| **AVG, MAX y MIN** | &nbsp; | <span style="color:#1F4E9C">¿Qué información da el promedio que no da el total? ¿Qué agregan el máximo y el mínimo?</span> |

<h2 style="color:#1E3A5F">Actividad 2 — Desarrollo práctico y ejecución</h2>

<h3 style="color:#1E3A5F">Caso de estudio: TecnoAndina Retail</h3>

TecnoAndina Retail es una cadena peruana de tecnología con tiendas en Lima, Arequipa, Trujillo y Piura, además de un canal de venta por Web. Comercializa doce productos agrupados en tres categorías —Cómputo, Celulares y Accesorios— y cada venta queda registrada en una tabla con la fecha, la sede, el canal, el producto, las unidades y el monto cobrado. Usted se incorpora como analista junior al área comercial y su primera semana coincide con la preparación de la campaña de fin de mes.

El problema aparece en la reunión semanal. La jefatura comercial debe decidir en qué sedes y categorías concentrará el presupuesto de la campaña, pero solo dispone del reporte de ventas del 3 al 8 de agosto de 2026, con 72 registros que nadie ha revisado: no se sabe cuáles fueron las ventas más altas, qué sedes venden más ni si vender más dinero significa que cada cliente compra con más valor. La jefatura lo plantea así: «Necesito saber dónde vendemos más, qué vendemos más caro y cuánto deja cada sede, antes de gastar un sol en la campaña». El encargo no consiste en predecir ni en explicar causas, sino en responder con consultas SQLite qué muestran las ventas registradas y qué información adicional haría falta para decidir.

> **Naturaleza de los datos.** Los datos de TecnoAndina Retail son simulados y de uso exclusivamente académico: no corresponden a una empresa real ni a registros oficiales. Cubren solo seis días, de modo que sirven para practicar consultas, no para afirmar tendencias ni causas del negocio completo.

**Lo que se le pide:**

1. Reconocer la estructura de la tabla ventas.
2. Filtrar ventas por sede, canal y monto, combinando condiciones con AND, OR y NOT.
3. Ordenar y limitar los resultados para identificar las ventas más altas y más bajas.
4. Listar las sedes y categorías existentes sin repeticiones.
5. Resumir el ingreso con COUNT, SUM, AVG, MAX y MIN.
6. Comparar dos sedes y redactar una recomendación prudente para la jefatura.

**Qué significa cada columna de la tabla ventas:**

| Columna | Qué significa |
|---|---|
| `venta_id` | Número correlativo que identifica cada venta. |
| `fecha_venta` | Día de la venta en formato AAAA-MM-DD, del 2026-08-03 al 2026-08-08. |
| `sede` | Lugar donde se registró la venta: Lima, Arequipa, Trujillo o Piura. |
| `canal` | Tienda (venta presencial) o Web (venta por internet). |
| `producto` | Nombre del producto vendido; hay doce productos distintos. |
| `categoria` | Grupo del producto: Cómputo, Celulares o Accesorios. |
| `unidades` | Cantidad de unidades vendidas en esa venta (de 1 a 3). |
| `precio_unitario` | Precio cobrado por cada unidad, en soles; varía levemente por promociones. |
| `monto` | Total cobrado en la venta, en soles: unidades × precio_unitario. |

<h3 style="color:#1E3A5F">Preparación del entorno</h3>

Abra el notebook en Google Colab y ejecute la primera celda. Esta crea la tabla ventas en SQLite con los 72 registros del caso y define la función mostrar_consulta(), que ejecuta una consulta y muestra su resultado. No modifique esta celda; el código se reproduce aquí para seguir el procedimiento aun sin abrir el notebook.

In [1]:
import sqlite3, random
# Datos simulados de TecnoAndina Retail: 72 ventas del 3 al 8 de agosto de 2026.
conexion = sqlite3.connect(":memory:")
conexion.execute("""CREATE TABLE ventas (
    venta_id INTEGER PRIMARY KEY,
    fecha_venta TEXT NOT NULL,
    sede TEXT NOT NULL,
    canal TEXT NOT NULL,
    producto TEXT NOT NULL,
    categoria TEXT NOT NULL,
    unidades INTEGER NOT NULL,
    precio_unitario REAL NOT NULL,
    monto REAL NOT NULL
)""")
productos = [
    ("Laptop 14 pulgadas", "Cómputo", 2400), ("Monitor 24 pulgadas", "Cómputo", 650),
    ("Teclado mecánico", "Cómputo", 180), ("Mouse inalámbrico", "Cómputo", 55),
    ("Smartphone gama media", "Celulares", 1200), ("Smartphone gama alta", "Celulares", 3200),
    ("Tablet 10 pulgadas", "Celulares", 900), ("Audífonos Bluetooth", "Accesorios", 160),
    ("Cargador rápido", "Accesorios", 70), ("Funda protectora", "Accesorios", 35),
    ("Memoria USB 64 GB", "Accesorios", 40), ("Parlante portátil", "Accesorios", 220)
]
sedes = ("Lima", "Arequipa", "Trujillo", "Piura")
azar = random.Random(4414)          # semilla fija: todos obtienen los mismos datos
registros = []#lista vacia
for dia in range(3, 9):
    for producto, categoria, precio in productos:
        unidades = azar.choice((1, 1, 2, 3)) if precio < 500 else azar.choice((1, 1, 1, 2, 3))
        sede = azar.choice(sedes)
        canal = "Web" if azar.random() < 0.25 else "Tienda"
        precio_cobrado = round(precio * (1 + azar.randint(-6, 6) / 100))
        registros.append((f"2026-08-{dia:02d}", sede, canal, producto, categoria,
                          unidades, precio_cobrado, float(precio_cobrado * unidades)))
conexion.executemany("""INSERT INTO ventas
    (fecha_venta, sede, canal, producto, categoria, unidades, precio_unitario, monto)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)""", registros)
conexion.commit()

def mostrar_consulta(sql):
    cursor = conexion.execute(sql)
    encabezados = [columna[0] for columna in cursor.description]
    filas = cursor.fetchall()
    print(" | ".join(encabezados))
    for fila in filas:
        print(" | ".join(str(valor) for valor in fila))
    print(f"Filas: {len(filas)}")
    return filas

print(f"Caso preparado: {len(registros)} ventas simuladas de TecnoAndina Retail.")

Caso preparado: 72 ventas simuladas de TecnoAndina Retail.


<h3 style="color:#2E74B5">A. Explorar y filtrar</h3>

<h3 style="color:#1E3A5F">Ejercicio 1 — Reconocer la estructura de la tabla</h3>

Antes de responder a la jefatura, usted necesita saber qué contiene el reporte. Una revisión rápida consiste en mirar las primeras filas y fijarse en los nombres de las columnas y en el tipo de valor que guarda cada una.

Ejecute la celda: mostrará las cinco primeras ventas con todas sus columnas y, luego, las ocho primeras con solo cuatro columnas.

In [2]:
import pandas as pd
consulta1 = """SELECT *
FROM ventas
LIMIT 5;"""
resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

,venta_id,fecha_venta,sede,canal,producto,categoria,unidades,precio_unitario,monto
0,1,2026-08-03,Piura,Tienda,Laptop 14 pulgadas,Cómputo,1,2256.0,2256.0
1,2,2026-08-03,Piura,Tienda,Monitor 24 pulgadas,Cómputo,1,637.0,637.0
2,3,2026-08-03,Trujillo,Tienda,Teclado mecánico,Cómputo,2,173.0,346.0
3,4,2026-08-03,Lima,Tienda,Mouse inalámbrico,Cómputo,3,53.0,159.0
4,5,2026-08-03,Trujillo,Tienda,Smartphone gama media,Celulares,1,1212.0,1212.0


In [3]:
consulta2 = """SELECT fecha_venta, sede, producto, monto
FROM ventas
LIMIT 8;"""
resultado2=pd.read_sql(consulta2,conexion)
display(resultado2)

,fecha_venta,sede,producto,monto
0,2026-08-03,Piura,Laptop 14 pulgadas,2256.0
1,2026-08-03,Piura,Monitor 24 pulgadas,637.0
2,2026-08-03,Trujillo,Teclado mecánico,346.0
3,2026-08-03,Lima,Mouse inalámbrico,159.0
4,2026-08-03,Trujillo,Smartphone gama media,1212.0
5,2026-08-03,Lima,Smartphone gama alta,6144.0
6,2026-08-03,Lima,Tablet 10 pulgadas,2619.0
7,2026-08-03,Piura,Audífonos Bluetooth,165.0


<span style="color:#1F4E9C"><b>Pregunta 1.</b> ¿Cuántas columnas tiene la tabla ventas y qué representa cada fila? Clasifique tres columnas como numéricas o de texto.</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> La tabla tiene 8 columnas. Cada fila representa una venta. venta_id es numérica, monto es numérica y sede es de texto

<span style="color:#1F4E9C"><b>Pregunta 2.</b> ¿Qué cláusula decidió que solo se mostraran cinco y ocho filas? ¿En qué se diferencia SELECT * de nombrar las columnas?</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> La cláusula LIMIT limita las filas mostradas. SELECT * muestra todas las columnas, mientras que nombrarlas permite seleccionar solo las necesarias.

<h3 style="color:#1E3A5F">Ejercicio 2 — Ventas de alto valor de una sede</h3>

La gerencia de la sede Lima quiere conocer qué ventas fueron de alto valor, es decir, de al menos S/ 1 000, para decidir qué productos destacar en la campaña. Para que una venta aparezca en el resultado debe cumplir dos condiciones al mismo tiempo: haberse registrado en Lima y tener un monto de 1 000 o más.

Ejecute la celda y observe cómo se combinan las dos condiciones en el WHERE.

In [20]:
consulta1 = """SELECT producto, sede, monto
FROM ventas
WHERE sede = 'Lima' OR monto >= 1000;"""

resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

,producto,sede,monto
0,Laptop 14 pulgadas,Piura,2256.0
1,Mouse inalámbrico,Lima,159.0
2,Smartphone gama media,Trujillo,1212.0
3,Smartphone gama alta,Lima,6144.0
4,Tablet 10 pulgadas,Lima,2619.0
5,Laptop 14 pulgadas,Piura,2280.0
6,Monitor 24 pulgadas,Lima,670.0
7,Smartphone gama media,Lima,1128.0
8,Smartphone gama alta,Piura,3392.0
9,Tablet 10 pulgadas,Lima,2754.0


<span style="color:#1F4E9C"><b>Pregunta 3.</b> ¿Por qué la consulta usa AND y no OR? Prediga cuántas filas devolvería si se cambiara AND por OR y luego compruébelo.</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> AND exige que la venta sea de Lima y tenga un monto de al menos S/ 1,000. Con AND devuelve 6 filas; con OR, 39 fila

<span style="color:#1F4E9C"><b>Pregunta 4.</b> ¿Qué productos aparecen entre las ventas de alto valor de Lima? ¿Qué puede afirmarse y qué no con solo seis días de datos?</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Aparecen Smartphone gama alta, Smartphone gama media, Tablet 10 pulgadas y Laptop 14 pulgadas. Solo podemos describir las ventas de los seis días analizados, no afirmar tendencias generales

<h3 style="color:#1E3A5F">Ejercicio 3 — Ventas relevantes de una sede: Piura</h3>

La jefa de la sede Piura necesita saber qué ventas superaron S/ 800 en su sede, porque quiere revisar el stock de esos productos antes de la campaña. Piura es la sede con menos registros y su jefa sospecha que vende poco pero de valor alto. La consulta ya está escrita: pide solo el producto, las unidades y el monto de esas ventas.

Ejecute la celda y lea con atención las dos condiciones del WHERE.

In [5]:
consulta1 = """SELECT producto, unidades, monto
FROM ventas
WHERE sede = 'Piura' AND monto > 800;"""
resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

,producto,unidades,monto
0,Laptop 14 pulgadas,1,2256.0
1,Laptop 14 pulgadas,1,2280.0
2,Smartphone gama alta,1,3392.0
3,Smartphone gama alta,1,3232.0
4,Monitor 24 pulgadas,2,1312.0
5,Smartphone gama alta,1,3008.0


<span style="color:#1F4E9C"><b>Pregunta 5.</b> Lea la consulta antes de interpretar el resultado: ¿qué dos condiciones aplica el WHERE? Luego indique cuántas filas devolvió y cuál fue la venta de mayor monto.</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Las condiciones son sede Piura y monto superior a S/ 800. Devuelve 6 filas. La venta de mayor monto es Smartphone gama alta, por S/ 3,392.

<h3 style="color:#1E3A5F">Ejercicio 4 — Alternativas y exclusiones con OR y NOT</h3>

El área de marketing prepara una acción que cubrirá solo Arequipa y Trujillo, y necesita las ventas de Celulares de esas dos sedes. Además, el área de comercio electrónico pide separar todas las ventas que no se hicieron en tienda para estudiarlas por separado.

Ejecute la celda: la primera consulta combina OR con AND mediante paréntesis; la segunda usa NOT.

In [6]:
consulta1 = """SELECT sede, producto, monto
FROM ventas
WHERE (sede = 'Arequipa' OR sede = 'Trujillo') AND categoria = 'Celulares';"""
resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

consulta1 = """SELECT sede, canal, producto, monto
FROM ventas
WHERE NOT canal = 'Tienda';"""
resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

,sede,producto,monto
0,Trujillo,Smartphone gama media,1212.0
1,Trujillo,Smartphone gama media,2304.0
2,Arequipa,Tablet 10 pulgadas,2862.0
3,Arequipa,Smartphone gama media,1224.0
4,Arequipa,Smartphone gama alta,3264.0
5,Trujillo,Tablet 10 pulgadas,855.0
6,Trujillo,Smartphone gama media,1248.0
7,Trujillo,Smartphone gama alta,6208.0
8,Trujillo,Tablet 10 pulgadas,2673.0


,sede,canal,producto,monto
0,Piura,Web,Audífonos Bluetooth,165.0
1,Trujillo,Web,Funda protectora,34.0
2,Arequipa,Web,Memoria USB 64 GB,84.0
3,Lima,Web,Monitor 24 pulgadas,670.0
4,Arequipa,Web,Teclado mecánico,187.0
5,Lima,Web,Smartphone gama media,1128.0
6,Lima,Web,Tablet 10 pulgadas,2754.0
7,Piura,Web,Audífonos Bluetooth,157.0
8,Piura,Web,Smartphone gama alta,3232.0
9,Trujillo,Web,Cargador rápido,67.0


<span style="color:#1F4E9C"><b>Pregunta 6.</b> En la primera consulta, ¿para qué sirven los paréntesis? ¿Qué pasaría si se quitaran?</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Los paréntesis agrupan las condiciones de Arequipa o Trujillo antes de aplicar el filtro de categoría Celulares. Sin ellos, la consulta podría incluir ventas de otras categorías registradas en Trujillo

<span style="color:#1F4E9C"><b>Pregunta 7.</b> ¿Qué hizo NOT en la segunda consulta? Escriba otra forma de expresar la misma condición.</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> NOT excluye las ventas realizadas en tienda y muestra las ventas Web. La condición equivalente es canal = 'Web'

<h3 style="color:#2E74B5">B. Ordenar, limitar y distinguir</h3>

<h3 style="color:#1E3A5F">Ejercicio 5 — Las ventas más altas y más bajas</h3>

Para la reunión, la jefatura pide dos listas cortas: las cinco ventas de mayor monto, para saber qué productos generan los tickets más altos, y las cinco de menor monto, para saber qué se vende en importes pequeños. En ambos casos el resultado debe ordenarse antes de limitarse.

Ejecute la celda y compare las dos listas.

In [7]:
consulta1 = """SELECT sede, producto, monto
FROM ventas
ORDER BY monto DESC
LIMIT 5;"""
resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

consulta1 = """SELECT sede, producto, monto
FROM ventas
ORDER BY monto ASC
LIMIT 5;"""
resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

,sede,producto,monto
0,Arequipa,Laptop 14 pulgadas,7416.0
1,Arequipa,Laptop 14 pulgadas,7128.0
2,Trujillo,Smartphone gama alta,6208.0
3,Lima,Smartphone gama alta,6144.0
4,Trujillo,Laptop 14 pulgadas,4512.0


,sede,producto,monto
0,Trujillo,Funda protectora,34.0
1,Trujillo,Funda protectora,37.0
2,Trujillo,Memoria USB 64 GB,39.0
3,Arequipa,Memoria USB 64 GB,39.0
4,Piura,Memoria USB 64 GB,42.0


<span style="color:#1F4E9C"><b>Pregunta 8.</b> ¿Qué productos y qué sedes aparecen entre las cinco ventas de mayor monto? ¿Y entre las cinco de menor monto?</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Cinco ventas más altas: Laptop 14 pulgadas en Arequipa (S/ 7,416 y S/ 7,128); Smartphone gama alta en Trujillo (S/ 6,208) y Lima (S/ 6,144); Laptop 14 pulgadas en Trujillo (S/ 4,512). Cinco ventas más bajas: Funda protectora en Trujillo (S/ 34 y S/ 37); Memoria USB 64 GB en Trujillo (S/ 39), Arequipa (S/ 39) y Piura (S/ 42).

<span style="color:#1F4E9C"><b>Pregunta 9.</b> ¿Por qué ORDER BY debe escribirse antes de LIMIT? ¿Qué devolvería LIMIT 5 si no se ordenara?</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> ORDER BY ordena los registros antes de aplicar LIMIT. Sin ordenamiento, LIMIT 5 mostraría las primeras cinco filas disponibles, no necesariamente las ventas más altas o bajas

<h3 style="color:#1E3A5F">Ejercicio 6 — Las tres mayores ventas por Web</h3>

El área de comercio electrónico quiere saber cuáles fueron las tres ventas por Web de mayor monto, porque con ellas decidirá qué productos mostrar en la portada de la tienda virtual durante la campaña. La consulta filtra por canal, ordena el monto de mayor a menor y limita el resultado a tres filas.

Ejecute la celda y observe cómo se combinan el filtro, el orden y el límite.

In [8]:
consulta1 = """SELECT producto, sede, monto
FROM ventas
WHERE canal = 'Web'
ORDER BY monto DESC
LIMIT 3;"""
resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

,producto,sede,monto
0,Laptop 14 pulgadas,Trujillo,4512.0
1,Smartphone gama alta,Piura,3232.0
2,Tablet 10 pulgadas,Lima,2754.0


<span style="color:#1F4E9C"><b>Pregunta 10.</b> Lea la consulta e identifique qué filtro, qué orden y qué límite aplica. Luego anote los tres productos con su sede y su monto.</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> El filtro es WHERE canal = 'Web', el orden es ORDER BY monto DESC y el límite es LIMIT 3.

- Laptop 14 pulgadas — Trujillo — S/ 4,512.

- Smartphone gama alta — Piura — S/ 3,232.

- Tablet 10 pulgadas — Lima — S/ 2,754.

<span style="color:#1F4E9C"><b>Pregunta 11.</b> Si el área Web decidiera promocionar solo estos tres productos porque aparecen en la lista, ¿qué riesgo correría?</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Se podrían ignorar productos con potencial de venta que no aparecen entre los tres primeros. Se necesita analizar más ventas y demanda antes de decidir.

<h3 style="color:#1E3A5F">Ejercicio 7 — Valores únicos con DISTINCT</h3>

Antes de armar el informe comparativo, usted quiere comprobar qué sedes y qué categorías existen realmente en la tabla, sin repeticiones. Es una verificación habitual: un error de digitación, por ejemplo «lima» en minúscula, crearía una sede fantasma y haría que un filtro devuelva cero filas.

Ejecute la celda: cada consulta muestra los valores distintos de una columna, ordenados alfabéticamente.

In [9]:
consulta1 = """SELECT DISTINCT sede
FROM ventas
ORDER BY sede;"""
resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

consulta1 = """SELECT DISTINCT categoria
FROM ventas
ORDER BY categoria;"""
resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

,sede
0,Arequipa
1,Lima
2,Piura
3,Trujillo


,categoria
0,Accesorios
1,Celulares
2,Cómputo


<span style="color:#1F4E9C"><b>Pregunta 12.</b> ¿Cuántas sedes y cuántas categorías distintas aparecen? ¿Coinciden con las que describe el caso?</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Hay 4 sedes: Arequipa, Lima, Piura y Trujillo; y 3 categorías: Accesorios, Celulares y Cómputo. Sí, coinciden con el caso

<span style="color:#1F4E9C"><b>Pregunta 13.</b> Sin DISTINCT, la consulta SELECT sede FROM ventas devolvería 72 filas. ¿Por qué? ¿Para qué sirve conocer los valores únicos antes de analizar?</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Sin DISTINCT, se mostrarían los 72 registros, incluidas las sedes repetidas. Los valores únicos permiten identificar las sedes y categorías reales y detectar errores de digitación

<h3 style="color:#2E74B5">C. Resumir para decidir</h3>

<h3 style="color:#1E3A5F">Ejercicio 8 — Contar y sumar: la sede Lima</h3>

La jefatura pregunta cuántas ventas registró la sede Lima y cuánto dinero ingresó en total, porque en Lima se concentrará la mayor parte de la inversión de la campaña. Son dos cifras distintas: una cuenta registros y la otra suma montos, y no deben confundirse en el informe.

Ejecute la celda y anote las dos cifras: se usarán en la Actividad 3.

In [10]:
consulta1 = """SELECT COUNT(*) AS numero_ventas, SUM(monto) AS ingreso_lima
FROM ventas
WHERE sede = 'Lima';"""
resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

,numero_ventas,ingreso_lima
0,20,20580.0


<span style="color:#1F4E9C"><b>Pregunta 14.</b> Explique con sus palabras qué significa cada cifra y por qué COUNT(*) no puede reemplazar a SUM(monto).</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> COUNT(*) indica que Lima registró 20 ventas, SUM(monto) indica ingresos de S/ 20,580. No son equivalentes porque uno cuenta registros y el otro suma dinero

<span style="color:#1F4E9C"><b>Pregunta 15.</b> ¿Puede una sede tener más ventas que otra y, aun así, ingresar menos dinero? Explique con un ejemplo del caso.</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Sí. Por ejemplo Lima registró 20 ventas y obtuvo S/ 20,580, mientras que Piura registró 13 ventas y obtuvo S/ 16,830. Esto demuestra que una sede con menos ventas puede generar ingresos importantes si vende productos de mayor valor

<h3 style="color:#1E3A5F">Ejercicio 9 — Promedio, máximo y mínimo de la cadena</h3>

Para tener una referencia general, la jefatura quiere conocer el monto promedio de una venta de la cadena, la venta más alta y la más baja de la semana. Las tres cifras juntas muestran qué tan dispersos son los montos: no es lo mismo una cadena donde todas las ventas se parecen que una donde conviven fundas de decenas de soles con laptops de miles.

Ejecute la celda y anote las tres cifras.

In [11]:
consulta1 = """SELECT ROUND(AVG(monto), 2) AS promedio, MAX(monto) AS maximo, MIN(monto) AS minimo
FROM ventas;"""
resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

,promedio,maximo,minimo
0,1201.36,7416.0,34.0


<span style="color:#1F4E9C"><b>Pregunta 16.</b> ¿Cuál es el monto promedio, el máximo y el mínimo? ¿Qué indica la distancia entre el máximo y el mínimo?</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Promedio: S/ 1,201.36. Máximo: S/ 7,416. Mínimo: S/ 34. La diferencia muestra una gran variación entre los montos de las ventas

<span style="color:#1F4E9C"><b>Pregunta 17.</b> ¿El promedio describe bien una venta típica de la cadena? Justifique con lo observado.</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> El promedio sirve como referencia general, pero no representa necesariamente una venta típica porque los productos económicos y los de alto precio generan montos muy diferentes

<h3 style="color:#1E3A5F">Ejercicio 10 — Resumen de la sede Piura</h3>

La jefa de Piura pide un resumen de su sede: cuántas ventas tuvo, cuánto ingresó y cuánto dejó en promedio cada venta. Con esas tres cifras podrá compararse con Lima en la Actividad 3.

Ejecute la celda y anote las tres cifras.

In [12]:
consulta1 = """SELECT COUNT(*) AS numero_ventas, SUM(monto) AS ingreso, ROUND(AVG(monto), 2) AS promedio
FROM ventas
WHERE sede = 'Piura';"""
resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

,numero_ventas,ingreso,promedio
0,13,16830.0,1294.62


<span style="color:#1F4E9C"><b>Pregunta 18.</b> Anote las tres cifras de Piura y explique qué significa cada una.</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span>

- Número de ventas: 13.

- Ingreso total: S/ 16,830.

- Promedio por venta: S/ 1,294.62.

Estas cifras indican cuántas ventas realizó Piura, cuánto dinero ingresó y cuánto se cobró en promedio por operación

<h2 style="color:#1E3A5F">Actividad 3 — Informe analítico comparado</h2>

Con los resultados de los ejercicios 8 y 10, elabore el informe comparado de las sedes Lima y Piura que espera la jefatura. Ejecute la celda siguiente para obtener, además, la venta más alta y la más baja de cada sede, complete la tabla y responda en términos de negocio, citando cifras del notebook.

In [21]:
consulta1 = """SELECT COUNT(*) AS numero_ventas, SUM(monto) AS ingreso, ROUND(AVG(monto), 2) AS promedio,
       MAX(monto) AS venta_mas_alta, MIN(monto) AS venta_mas_baja
FROM ventas
WHERE sede = 'Lima';"""
resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

,numero_ventas,ingreso,promedio,venta_mas_alta,venta_mas_baja
0,20,20580.0,1029.0,6144.0,53.0


In [22]:

consulta1 = """SELECT COUNT(*) AS numero_ventas, SUM(monto) AS ingreso, ROUND(AVG(monto), 2) AS promedio,
       MAX(monto) AS venta_mas_alta, MIN(monto) AS venta_mas_baja
FROM ventas
WHERE sede = 'Piura';"""
resultado1=pd.read_sql(consulta1,conexion)
display(resultado1)

,numero_ventas,ingreso,promedio,venta_mas_alta,venta_mas_baja
0,13,16830.0,1294.62,3392.0,42.0


| Indicador | Lima | Piura | ¿Cuál es mayor? |
|---|---|---|---|
| **Número de ventas** | &nbsp;20 | &nbsp; 13| &nbsp;Lima|
| **Ingreso total (S/)** | &nbsp;20 580 | &nbsp;16830 | &nbsp;Lima |
| **Monto promedio por venta (S/)** | &nbsp;1029 | &nbsp;1294.62 | &nbsp;Piura |
| **Venta más alta (S/)** | &nbsp; 6144| &nbsp;3392 | &nbsp;Lima |
| **Venta más baja (S/)** | &nbsp;53 | &nbsp; 42| &nbsp;Lima |

<span style="color:#1F4E9C"><b>A.</b> ¿Qué sede aporta más ingreso y cuál tiene el monto promedio por venta más alto? ¿Por qué no coinciden?</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Lima aporta más ingres (S/ 20580 frente a S/ 16830) porque registro más ventas. Piura tiene el monto promedio más alto. No coinciden porque el ingreso depende de cuántas ventas hay y el promedio, de cuánta vale cada una.

<span style="color:#1F4E9C"><b>B.</b> ¿Qué consulta de este laboratorio le permitiría verificar si la diferencia en el monto promedio se debe al tipo de productos que vende cada sede? Escríbala o descríbala.</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> ______________________________________________

<span style="color:#1F4E9C"><b>C.</b> ¿Qué datos adicionales pediría antes de decidir en qué sede invertir el presupuesto de la campaña? Mencione al menos tres.</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Solicitaría datos de costos y márgenes de ganancia, inventario disponible y ventas de varios meses. También revisaría los gastos de campaña y la demanda por sede

<span style="color:#1F4E9C"><b>D.</b> Redacte en tres líneas una recomendación prudente para la jefatura, con cifras y señalando qué no puede afirmarse.</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Lima registra ingresos de S/ 20,580 y Piura de S/ 16,830; sin embargo, Piura tiene un promedio por venta superior.
Recomendaria comparar la rentabilidad, el inventario y la demanda antes de distribuir el presupuesto.
Los seis días analizados no permiten afirmar que estos resultados representen el comportamiento habitual de las sedes.

<h2 style="color:#1E3A5F">Reto de aplicación y retroalimentación</h2>

En esta sección se aplican los procedimientos desarrollados durante la sesión a nuevas situaciones del mismo caso. Cada reto presenta la consulta ya escrita sobre la tabla ventas: el estudiante la ejecuta, lee qué pide cada cláusula e interpreta el resultado desde una perspectiva comercial. El propósito es comprobar la lectura correcta de las consultas y la capacidad de convertir un resultado en una recomendación.

**Indicaciones generales.** Los retos operan sobre la tabla ventas, ya disponible en el notebook. Cada respuesta escrita no debe exceder cuatro líneas. Los retos se ejecutan e interpretan en el notebook y quedan como evidencia en el repositorio de GitHub.

<h3 style="color:#1E3A5F">Reto 1 — Tickets más altos de Cómputo en Trujillo</h3>

Trujillo es la sede que más ventas registró en la semana (22 de 72). Su jefatura quiere revisar las tres ventas más altas de la categoría Cómputo para decidir si reserva más laptops y monitores para la campaña. Un ticket alto señala qué productos de valor se están moviendo en la sede, aunque no basta por sí solo para asegurar el stock.

**Se solicita:**

1. Ejecutar la consulta de la celda, que filtra las ventas de la categoría Cómputo registradas en la sede Trujillo, muestra producto, unidades y monto, las ordena de mayor a menor monto y limita el resultado a tres filas.
2. Identificar qué cláusula cumple cada una de esas funciones.
3. Interpretar los tres productos y montos obtenidos.

In [25]:
consulta = """SELECT producto, unidades, monto
FROM ventas
WHERE categoria = 'Cómputo' AND sede = 'Trujillo'
ORDER BY monto DESC
LIMIT 3;"""
mostrar_consulta(consulta)

producto | unidades | monto
Laptop 14 pulgadas | 2 | 4512.0
Monitor 24 pulgadas | 2 | 1364.0
Teclado mecánico | 2 | 346.0
Filas: 3


[('Laptop 14 pulgadas', 2, 4512.0),
 ('Monitor 24 pulgadas', 2, 1364.0),
 ('Teclado mecánico', 2, 346.0)]

<span style="color:#1F4E9C"><b>Pregunta de cierre.</b> ¿Qué productos y montos obtuvo? ¿Qué decisión de abastecimiento sugiere el resultado y qué advertencia haría por el periodo analizado?</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Los resultados son Laptop 14 pulgadas (2 unidades, S/ 4,512), Monitor 24 pulgadas (2 unidades, S/ 1,364) y Teclado mecánico (2 unidades, S/ 346). Mi recomendación sería revisar el stock de laptops y monitores, sin asumir que la demanda futura será igual a la de estos seis días

<h3 style="color:#1E3A5F">Reto 2 — Accesorios vendidos por la Web</h3>

El área de comercio electrónico duda de si vale la pena promocionar accesorios en la tienda virtual: cree que se venden muchos, pero a precios tan bajos que dejan poco. Para discutirlo con cifras necesita saber cuántas ventas de Accesorios se hicieron por Web, cuánto ingresaron en total y cuál fue el monto promedio de cada una.

**Se solicita:**

1. Ejecutar la consulta de la celda, que filtra las ventas de la categoría Accesorios hechas por el canal Web y obtiene el número de ventas, el ingreso total y el monto promedio con COUNT, SUM y AVG.
2. Distinguir qué mide cada una de las tres cifras (los alias ventas, ingreso y promedio nombran las columnas).
3. Interpretar el resultado desde la pregunta del área de comercio electrónico.

In [26]:
consulta = """SELECT COUNT(*) AS ventas, SUM(monto) AS ingreso, ROUND(AVG(monto), 2) AS promedio
FROM ventas
WHERE categoria = 'Accesorios' AND canal = 'Web';"""
mostrar_consulta(consulta)

ventas | ingreso | promedio
9 | 1094.0 | 121.56
Filas: 1


[(9, 1094.0, 121.56)]

<span style="color:#1F4E9C"><b>Pregunta de cierre.</b> ¿Cuántas ventas, qué ingreso y qué promedio obtuvo? Con esas cifras, ¿recomendaría promocionar accesorios en la web? ¿Qué dato adicional pediría?</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Se registraron 9 ventas, con ingresos de S/ 1,094 y un promedio de S/ 121.56 por venta. Recomendaría evaluar una campaña de accesorios, pero antes solicitaría los costos, márgenes de ganancia y tasas de conversión del canal Web

<h3 style="color:#1E3A5F">Reto 3 — Celulares frente a Accesorios</h3>

La jefatura debe repartir el presupuesto de la campaña entre categorías y tiene una intuición: «los accesorios se venden más, así que debemos invertir más en ellos». Usted debe contrastar esa intuición con los datos, porque vender más veces no es lo mismo que ingresar más dinero. Para ello comparará la categoría Celulares con la categoría Accesorios.

**Se solicita:**

1. Ejecutar la celda, que contiene dos consultas, una por categoría (Celulares y Accesorios), con WHERE y las funciones COUNT, SUM y AVG.
2. Comparar los resultados: qué categoría tiene más ventas y cuál más ingreso.
3. Contrastar el resultado con la intuición de la jefatura.

In [17]:
consulta = """SELECT COUNT(*) AS ventas, SUM(monto) AS ingreso, ROUND(AVG(monto), 2) AS promedio
FROM ventas
WHERE categoria = 'Celulares';"""
mostrar_consulta(consulta)

consulta = """SELECT COUNT(*) AS ventas, SUM(monto) AS ingreso, ROUND(AVG(monto), 2) AS promedio
FROM ventas
WHERE categoria = 'Accesorios';"""
mostrar_consulta(consulta)

ventas | ingreso | promedio
18 | 46242.0 | 2569.0
Filas: 1
ventas | ingreso | promedio
30 | 5454.0 | 181.8
Filas: 1


[(30, 5454.0, 181.8)]

<span style="color:#1F4E9C"><b>Pregunta de cierre.</b> ¿Qué categoría tiene más ventas y cuál más ingreso? ¿Confirma el resultado la intuición de la jefatura? Formule una recomendación prudente e indique qué dato adicional pediría.</span>

<span style="color:#1E3A5F"><b>Respuesta:</b></span> Celulares registró 18 ventas, ingresos de S/ 46,242 y un promedio de S/ 2,569. Accesorios registró 30 ventas, ingresos de S/ 5,454 y un promedio de S/ 181.80. La intuición de que los accesorios venden más veces es correcta, pero no significa que generen más ingresos. Recomendaría distribuir el presupuesto considerando también los márgenes de ganancia, la demanda y el inventario

<h2 style="color:#1E3A5F">Conclusiones</h2>

Escriba tres conclusiones técnicas y de negocio de este laboratorio.

- SQL permite filtrar, ordenar y resumir los registros para responder preguntas comerciales de forma organizada.

- La cantidad de ventas no equivale al ingreso total: Celulares generó más ingresos que Accesorios pese a registrar menos operaciones.

- Los datos de seis días permiten describir las ventas observadas, pero se necesita más información sobre rentabilidad, inventario y demanda para tomar decisiones de campaña.